# What actually couples the darts of a visit

Notebook 19 found the project's oldest assumption failing: the three darts of a visit are not
independent draws. Hitting the treble 20 raised the next dart's chance of doing the same by
18 points. This notebook asks the follow-up question, which is the one that changes the code:
**what should replace the assumption?**

The answer turns out to require unpicking that 18 points first, because it is not one effect.
Three separate things are mixed into it, and only the third is about the throw at all:

1. **The aim moves.** A player who misses the treble 20 often switches to the treble 19 for
   the rest of the visit. After a miss, the next dart is frequently not *aimed* at the treble
   20 -- so of course it hits it less often. This is a decision, and the model has no state
   for it.
2. **The tails are wrong.** A Gaussian tight enough to hit the treble 20 at a professional
   rate puts essentially nothing in the double 20 or off the board. Real players land there
   percents of the time.
3. **Whatever is left.** Once the aim and the tails are in the model, is there any coupling
   between darts remaining -- and if so, what shape?

The method is the same throughout: write the models down first, derive statistics that tell
them apart, check on simulated data that those statistics really do tell them apart, and only
then fit anything real -- on a training split, judged on held-out visits.

In [ ]:
import os
import sys
module_path = os.path.abspath(os.path.join('..', '..'))
if module_path not in sys.path:
    sys.path.append(module_path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25,
    'grid.linewidth': 0.6, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 11, 'font.size': 9, 'legend.frameon': False,
})

from darts.calibration import SCORING_FLOOR
from darts.dependence import (BedGrid, VisitModel, encode_visits, signatures,
                              treble_centre_mm, TARGETS)

DATA = os.path.join(module_path, 'data', 'real')
RES = os.path.join(module_path, 'results', 'dependence')
if not os.path.exists(os.path.join(DATA, 'per_dart.csv')):
    raise SystemExit('run scripts/build_real_data.py first (see data/real/README.md)')

GRID = BedGrid(512)
KEY = ['source', 'player', 'leg_id', 'visit_index']

per_dart = pd.read_csv(os.path.join(DATA, 'per_dart.csv'), low_memory=False)
d = per_dart[(per_dart.post_bust_visit == 0) & per_dart.dart_index.isin([1, 2, 3])]
info = d.groupby(KEY).agg(n=('dart_index', 'size'), start=('score_before', 'max'),
                          total=('value', 'sum')).reset_index()
# only visits that start high enough that the scoring filter cannot select on
# their outcome -- notebook 19 showed selection biases the dependence downward
ok = info[(info.n == 3) & ((info.start - info.total) >= SCORING_FLOOR)
          & (info.start >= SCORING_FLOOR + 180)]
scoring = d.merge(ok[KEY], on=KEY)
VIS = (scoring.pivot_table(index=KEY, columns='dart_index', values='bed',
                           aggfunc='first').dropna().reset_index())
print(f'{len(VIS):,} unfiltered pure-scoring visits, {VIS.player.nunique()} players')

## 1 · The aim moves

The tell is the treble 19. It sits eleven segments from the treble 20, on the opposite side of
the board -- no dart aimed at one can land in the other by accident. So a dart in the 19
neighbourhood is direct evidence of where it was *aimed*, which is the one thing a scoresheet
normally cannot give us.

In [ ]:
N20 = {'T20', 'S20', 'S5', 'S1', 'T5', 'T1', 'D20', 'D5', 'D1'}
N19 = {'T19', 'S19', 'S3', 'S7', 'T3', 'T7', 'D19', 'D3', 'D7'}
side = lambda b: np.where(b.isin(N20), '20', np.where(b.isin(N19), '19', 'other'))
for i in (1, 2, 3):
    VIS[f'side{i}'] = side(VIS[i])

print('share of darts thrown at the 19 neighbourhood, by position in the visit:')
for i in (1, 2, 3):
    print(f'   dart {i}: {(VIS[f"side{i}"] == "19").mean():.3f}')

print('\nwhere dart 2 goes, given how dart 1 went:')
for cond, label in [(VIS[1] == 'T20', 'dart 1 hit T20      '),
                    ((VIS.side1 == '20') & (VIS[1] != 'T20'), 'dart 1 missed, near 20'),
                    (VIS.side1 == '19', 'dart 1 was at the 19 ')]:
    g = VIS[cond]
    print(f'   {label}  n={len(g):6,}  ->  20: {(g.side2 == "20").mean():.3f}   '
          f'19: {(g.side2 == "19").mean():.3f}   other: {(g.side2 == "other").mean():.3f}')

*(aim rule interpretation filled in from the output)*

In [ ]:
def lift(a, b):
    h, m = b[a == 1].mean(), b[a == 0].mean()
    n1, n0 = int((a == 1).sum()), int((a == 0).sum())
    se = np.sqrt(h * (1 - h) / n1 + m * (1 - m) / n0)
    return 100 * (h - m), 100 * se

t20 = {i: (VIS[i] == 'T20').astype(float) for i in (1, 2, 3)}
treb = {i: VIS[i].str.startswith('T').astype(float) for i in (1, 2, 3)}
rows = [('hit T20 (what notebook 19 measured)', *lift(t20[1], t20[2])),
        ('hit ANY treble (target-invariant)', *lift(treb[1], treb[2]))]
pure = VIS[~(((VIS[['side1', 'side2', 'side3']] == '20').any(axis=1))
             & ((VIS[['side1', 'side2', 'side3']] == '19').any(axis=1)))]
rows.append(('hit T20, visits that never switch',
             *lift((pure[1] == 'T20').astype(float), (pure[2] == 'T20').astype(float))))
out = pd.DataFrame(rows, columns=['dart 1 -> dart 2', 'lift (pts)', 'se'])
out['z'] = out['lift (pts)'] / out.se
out.round(2)

*(target-invariant lift interpretation filled in from the output)*

### Does the switch cost them anything?

Notebook 18 established that above a remaining score of 250 the treble 20 is the optimal aim
for every ability the model covers -- that is what makes the whole scoring-phase analysis
possible. So the model says this switch is a mistake. The data can price it.

The comparison has to be made **within a player**, since players differ both in how often they
switch and in how well they throw. And it is observational, not causal: a player decides to
switch knowing things about the dart that just landed which the bed label does not record, so
what follows is what switching is *associated* with, not what it *causes*.

In [ ]:
from darts.dependence import bed_geometry

# Every bed gets a value and a target, so that nothing is dropped. Excluding the
# beds that sit near neither target would be selecting on dart 2's *outcome* --
# and the beds it would exclude are the worst darts, which is exactly the
# comparison being made. Instead each dart is assigned to whichever target it is
# angularly nearer, the same rule the signature statistics use.
def bed_value(name):
    if name == 'MISS':
        return 0
    if name == '25':
        return 25
    if name == 'BULL':
        return 50
    return int(name[1:]) * {'S': 1, 'D': 2, 'T': 3}[name[0]]

angle, _ = bed_geometry(GRID)
stack = np.stack([angle[t] for t in TARGETS])
nearest = np.argmin(np.where(np.isnan(stack), np.inf, np.abs(stack)), axis=0)
name_to_code = {n: i for i, n in enumerate(GRID.names)}
aimed_at = {n: TARGETS[nearest[i]] for n, i in name_to_code.items()}

missed = VIS[(VIS.side1 == '20') & (VIS[1] != 'T20')].copy()
missed['score2'] = missed[2].map(bed_value)
missed['aim2'] = missed[2].map(aimed_at)
assert missed.score2.notna().all() and missed.aim2.notna().all()

rows = []
for (src, pl), g in missed.groupby(['source', 'player']):
    stay, move = g[g.aim2 == 20], g[g.aim2 == 19]
    if min(len(stay), len(move)) < 40:
        continue
    diff = move.score2.mean() - stay.score2.mean()
    se = np.sqrt(move.score2.var() / len(move) + stay.score2.var() / len(stay))
    rows.append({'player': pl, 'n stay': len(stay), 'n move': len(move),
                 'stayed at 20': stay.score2.mean(), 'moved to 19': move.score2.mean(),
                 'diff': diff, 'se': se})
cost = pd.DataFrame(rows).sort_values('diff')
w = 1 / cost.se ** 2
print(f'{len(cost)} players with at least 40 of each')
print(f'pooled difference in dart-2 score, moving vs staying: '
      f'{(w * cost["diff"]).sum() / w.sum():+.2f} +/- {np.sqrt(1 / w.sum()):.2f} points')
print(f'players where moving scored less: {(cost["diff"] < 0).sum()} of {len(cost)}')
cost.round(2).head(10)

*(cost of switching interpretation filled in from the output)*

## 2 · The tails are wrong

The second thing the data forces into the model has nothing to do with dependence. Take the
player with the most clean scoring visits and look at where his *first* dart goes -- the one
dart we can be sure was aimed at the treble 20.

In [ ]:
who = VIS.groupby(['source', 'player']).size().idxmax()
sub = VIS[(VIS.source == who[0]) & (VIS.player == who[1])]
beds, hit = encode_visits(sub[[1, 2, 3]].values, GRID)
obs = np.bincount(beds[:, 0], minlength=GRID.n_beds) / len(beds)
top = np.argsort(obs)[::-1][:8]

rows = {'OBSERVED (dart 1)': obs[top]}
for sigma in (6.5, 8.0, 11.0, 14.5):
    rows[f'gaussian {sigma} mm'] = GRID.bed_pmf(20, np.zeros(2), sigma)[top]
tail = pd.DataFrame(rows, index=[GRID.names[i] for i in top]).T
print(f'{who[1]}: {len(beds):,} visits')
tail.round(4)

*(tails interpretation filled in from the output)*

In [ ]:
core = GRID.bed_pmf(20, np.zeros(2), 6.9)
wide = GRID.wide_pmf(20, 6.9 * 6.0)
mix = 0.87 * core + 0.13 * wide
comp = pd.DataFrame({'OBSERVED': obs[top], 'gaussian 6.9 mm': core[top],
                     '87% core + 13% wide': mix[top]},
                    index=[GRID.names[i] for i in top]).T
comp.round(4)

## 3 · The model family

Six models, each adding one mechanism to the last, so that every comparison isolates one
thing. All of them keep the throw isotropic: notebook 12 showed shape matters, but letting it
vary here would confound the question being asked.

| | mechanism added | what it says |
|---|---|---|
| **A** | -- | one target, Gaussian, independent darts. **What the project assumes today** |
| **B** | the aim rule | after a miss the player may move to the treble 19 |
| **C** | a wide component | a fraction `eps` of darts come from a much wider throw |
| **D** | a shared **offset** | the visit's aim point is drawn once: darts cluster in *direction* |
| **E** | a shared **scale** | the visit is tight or loose: darts agree in *magnitude* only |
| **F** | both | |

D and E are the two simple ways to break independence, and they are not the same story. D is
"he had that stance for those three darts". E is "he was in the groove that visit". They make
different, checkable predictions, which is the next section.

Everything is fitted to *bed sequences* by exact maximum likelihood. The per-visit latent is
integrated out by Gauss-Hermite quadrature; the latent target of each dart is summed out
exactly with a two-state forward pass, which is cheap because a visit is three darts long.

### The statistics that tell D from E

Pre-specified, before any fitting, and computed by one function applied identically to real
visits and to simulated ones.

* **`dir_corr`** -- the correlation between the *signed* angular offsets of two darts in a
  visit, measured in segments from whichever target they were thrown at. A shared offset
  pushes a whole visit one way round the board, so this is positive. A shared scale has no
  preferred direction, so this is zero.
* **`mag_corr`** -- the same for *absolute* offsets. Both models make this positive.

So the ratio separates them: D couples direction and magnitude about equally, E couples
magnitude only. Before trusting that, it has to be shown on data where the truth is known.

In [ ]:
rng = np.random.default_rng(0)
spec = {'M0 independent': dict(switching=True),
        'M1 shared offset': dict(shared_offset=True, switching=True),
        'M2 shared scale': dict(shared_scale=True, switching=True)}
truth = {'M0 independent': [np.log(8.6), 0.0, -3.5, -1.0],
         'M1 shared offset': [np.log(7.0), 0.0, np.log(5.0), -3.5, -1.0],
         'M2 shared scale': [np.log(8.35), 0.0, np.log(0.42), -3.5, -1.0]}

rows = []
for name, kw in spec.items():
    m = VisitModel(GRID, **kw)
    b, h = m.simulate(np.array(truth[name]), 20000, rng=rng)
    s = signatures(b, h, GRID)
    rows.append({'simulated from': name, 'P(T20)': s['p_t20'],
                 'T20 lift 1->2': s['t20_lift_12'],
                 'treble lift 1->2': s['treble_lift_12'],
                 'dir_corr': s['dir_corr'], 'mag_corr': s['mag_corr']})
pd.DataFrame(rows).set_index('simulated from').round(3)

*(signature validation interpretation filled in from the output)*

## 4 · The fits

`scripts/dependence_fits.py` fits all six models to every player with at least 400 clean
scoring visits. The split is on whole **legs**, not visits -- visits inside a leg share a
player, an evening and a scoreline, so splitting on visits would leak information across the
boundary. Both halves contain the same players by design: the question is whether a model
generalises to new visits, not to new people.

Models are scored by held-out log-likelihood per visit, which needs no penalty term because
the parameters were never fitted to the visits doing the scoring.

In [ ]:
# What model A -- the project's current assumption -- makes of real darts.
# With one fixed target it has no way to produce a dart at the 19 at all, so the
# question is not how well it fits but how much of the data it calls impossible.
impossible = np.mean([(VIS[f'side{i}'] == '19').mean() for i in (2, 3)])
p20 = GRID.bed_pmf(20, np.zeros(2), 13.1)
print('darts 2 and 3 landing in the 19 neighbourhood: %.1f%% of them' % (100 * impossible))
print('probability model A gives a dart at the S19: %.3g' % p20[GRID.names.index('S19')])
print('\nso model A is not merely a worse fit -- it assigns essentially zero')
print('probability to a fifth of the darts professionals actually throw.')

In [ ]:
fits = pd.read_csv(os.path.join(RES, 'fits.csv'))
sigs = pd.read_csv(os.path.join(RES, 'signatures.csv'))
print(f'{fits.player.nunique()} players, {fits.model.nunique()} models, '
      f'{fits.n_train.sum():,} training and {fits.n_test.sum():,} held-out visits')

# Model A is quoted separately below. Its log-likelihood is not a measure of
# fit but of impossibility: with no aim rule it gives every dart at the 19 a
# probability of zero, so its value is set by the numerical floor rather than by
# anything about the player. The ladder is therefore baselined on B.
base = (fits[fits.model == 'B + aim rule']
        .set_index(['source', 'player']).test_ll_per_visit)
fits['gain'] = fits.apply(
    lambda r: r.test_ll_per_visit - base.loc[(r.source, r.player)], axis=1)
ladder = fits.groupby('model').agg(
    players=('player', 'size'),
    gain=('gain', 'mean'),
    worst=('gain', 'min'),
    best=('gain', 'max'),
    sigma=('sigma', 'median'), tau=('tau', 'median'), nu=('nu', 'median'),
    eps=('eps', 'median'), s_miss=('s_miss', 'median')).round(4)
ladder

*(ladder interpretation filled in from the output)*

In [ ]:
step = {'C + wide tail': 'B + aim rule',
        'D + shared offset': 'C + wide tail', 'E + shared scale': 'C + wide tail',
        'F + both': 'E + shared scale'}
piv = fits.pivot_table(index=['source', 'player'], columns='model',
                       values='test_ll_per_visit')
rows = []
for model, prev in step.items():
    delta = piv[model] - piv[prev]
    rows.append({'step': f'{prev.split()[0]} -> {model.split()[0]}',
                 'adds': model.split(' ', 1)[1],
                 'mean gain': delta.mean(), 'median': delta.median(),
                 'players improved': f'{(delta > 0).sum()} / {len(delta)}'})
pd.DataFrame(rows).round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 4.0))
order = ['B + aim rule', 'C + wide tail',
         'D + shared offset', 'E + shared scale', 'F + both']
for (src, pl), g in fits.groupby(['source', 'player']):
    g = g.set_index('model').reindex(order)
    ax.plot(range(len(order)), g.gain.values, marker='o', ms=3, lw=0.8,
            alpha=0.45, color='#2b6cb0')
mean = fits.groupby('model').gain.mean().reindex(order)
ax.plot(range(len(order)), mean.values, marker='o', ms=7, lw=2.4,
        color='#dd6b20', label='mean over players', zorder=5)
ax.set_xticks(range(len(order)))
ax.set_xticklabels([o.replace(' + ', '\n+ ') for o in order], fontsize=8)
ax.set_ylabel('held-out log-likelihood per visit,\nabove the aim-rule model')
ax.set_title('Each step adds one mechanism')
ax.legend()
fig.tight_layout()

## 5 · Does the winning model reproduce what the data does?

A better likelihood is not the same as a model that behaves like a darts player. The check is
the pre-specified signatures: draw visits from each fitted model and ask whether they show
what the real visits show. Nothing here was fitted to these statistics.

In [ ]:
cols = ['p_t20', 't20_lift_12', 'treble_lift_12', 'dir_corr', 'mag_corr', 'switch_rate']
w = sigs.groupby(['model'])[cols].mean()
n_obs = sigs[sigs.model == 'OBSERVED'].set_index(['source', 'player']).n
tab = w.reindex(['OBSERVED'] + order)
tab.round(3)

*(predictive check interpretation filled in from the output)*

In [ ]:
obs_row = sigs[sigs.model == 'OBSERVED'].set_index(['source', 'player'])
fig, axes = plt.subplots(1, 3, figsize=(10.4, 3.4))
for ax, stat, title in zip(axes,
                           ['t20_lift_12', 'dir_corr', 'mag_corr'],
                           ['T20 lift, dart 1 -> 2 (pts)',
                            'direction coupling', 'magnitude coupling']):
    for model, colour in [('C + wide tail', '#a0aec0'), ('D + shared offset', '#dd6b20'),
                          ('E + shared scale', '#2b6cb0')]:
        m = sigs[sigs.model == model].set_index(['source', 'player'])
        common = obs_row.index.intersection(m.index)
        ax.scatter(obs_row.loc[common, stat], m.loc[common, stat], s=16,
                   alpha=0.75, color=colour, label=model.split(' ', 1)[1])
    lo = min(ax.get_xlim()[0], ax.get_ylim()[0])
    hi = max(ax.get_xlim()[1], ax.get_ylim()[1])
    ax.plot([lo, hi], [lo, hi], color='k', lw=0.8, ls=':')
    ax.set_xlabel('observed'); ax.set_ylabel('model'); ax.set_title(title)
axes[0].legend(fontsize=7)
fig.tight_layout()

In [ ]:
best = fits.loc[fits.groupby(['source', 'player']).test_ll_per_visit.idxmax()]
print('model chosen per player, by held-out likelihood:')
print(best.model.value_counts().to_string())

win = fits[fits.model == 'D + shared offset'].set_index(['source', 'player'])
print('\nfitted per-visit coupling, model D:')
show = win[['n_train', 'sigma', 'tau', 'eps', 'kappa', 's_hit', 's_miss']].copy()
show['sigma_total'] = np.hypot(show.sigma, show.tau)
show['tau share of variance'] = show.tau ** 2 / (show.sigma ** 2 + show.tau ** 2)
print(show.sort_values('n_train', ascending=False).head(12).round(3).to_string())

*(fitted parameters interpretation filled in from the output)*

## Verdict

*(filled in from the outputs above)*